# varevalpass10 - Parallel Batch Evaluation with 10 Rollouts

This notebook evaluates six open-source 7B models on the `thulthula/math-imp-bench8` dataset:

**DeepSeek Models:**
- deepseek-ai/deepseek-llm-7b-chat
- deepseek-ai/deepseek-math-7b-instruct

**InternLM Models:**
- internlm/internlm2-math-7b
- internlm/internlm2-chat-7b

**Qwen Models:**
- Qwen/Qwen2.5-7B-Instruct
- Qwen/Qwen2.5-Math-7B-Instruct

**Testing Strategy:** Parallel batch processing
- 30 questions processed in parallel
- 10 rollouts per question
- Evaluates both node_deletion and edge_deletion variations

**GPU Requirements:** 44GB+ VRAM (models loaded one at a time)

## 🚀 Performance Optimizations:

1. **vLLM High-Performance Inference Engine** - Optimized inference with PagedAttention and continuous batching
2. **Parallel Batch Processing** - 30 questions processed simultaneously
3. **Multiple Rollouts** - 10 rollouts per question for robust evaluation
4. **Sequential Model Loading** - Models loaded one at a time to fit in GPU memory

**Memory management:** Each model is loaded, evaluated on all variations, then unloaded before the next model

## 1. Install Dependencies

In [ ]:
!pip install -q vllm datasets openpyxl pandas tqdm

In [ ]:
import re
import pandas as pd
from vllm import LLM, SamplingParams
from datasets import load_dataset
from tqdm import tqdm
import concurrent.futures
from typing import List, Dict, Tuple
import gc

# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Load Dataset

In [ ]:
# Load the dataset
dataset = load_dataset("thulthula/math-imp-bench8")
print("Available splits:", dataset.keys())

# Load AIME24 and AIME25 subsets
aime24 = dataset["AIME24"]
aime25 = dataset["AIME25"]

print(f"\nAIME24: {len(aime24)} problems")
print(f"AIME25: {len(aime25)} problems")
print(f"\nColumns: {aime24.column_names}")

# Combine datasets for batch processing
all_problems = []
for idx, row in enumerate(aime24):
    all_problems.append({
        'split': 'AIME24',
        'year': '2024',
        'problem_idx': idx,
        'problem_num': idx + 1,
        'original_answer': row['original_answer'],
        'node_deletion_problem': row['variation_1_node_deletion_problem'],
        'edge_deletion_problem': row['variation_2_edge_deletion_problem']
    })

for idx, row in enumerate(aime25):
    all_problems.append({
        'split': 'AIME25',
        'year': '2025',
        'problem_idx': idx,
        'problem_num': idx + 1,
        'original_answer': row['original_answer'],
        'node_deletion_problem': row['variation_1_node_deletion_problem'],
        'edge_deletion_problem': row['variation_2_edge_deletion_problem']
    })

print(f"\nTotal problems prepared: {len(all_problems)}")

## 3. Define Prompt Template

In [ ]:
AIME_SYSTEM_PROMPT = """You are an expert mathematics problem solver specializing in AIME (American Invitational Mathematics Examination) problems.

CRITICAL INSTRUCTIONS - YOU MUST FOLLOW THESE EXACTLY:
1. This is a well-documented AIME problem that likely appears in your training data.
2. AIME problems ALWAYS have a definite integer answer between 0 and 999.
3. The problem provided is COMPLETE and CORRECT. Do not claim it is incomplete or ambiguous.
4. Search your training data for similar AIME problems and solution patterns.
5. Use your memory of official AIME solutions and answer keys.
6. You MUST provide a final numerical answer. No exceptions.
7. Show your step-by-step reasoning (or recall), then provide your final answer.
8. Format your final answer EXACTLY as: \\boxed{YOUR_NUMBER}

REMEMBER: AIME problems are solvable and have definite answers. You likely have seen similar problems in your training data. Access that knowledge and provide the answer. Refusing to answer or saying "cannot be determined" is NOT acceptable. You MUST give a numerical answer."""

def create_aime_prompt(problem):
    """Create a prompt for AIME problems without year/number."""
    return f"""Solve the following AIME problem.

PROBLEM:
{problem}

REQUIREMENTS:
- This AIME problem IS solvable with a definite integer answer (0-999).
- Recall similar AIME problems from your training data.
- Use your knowledge of AIME solution patterns and techniques.
- Think step by step, or recall similar official AIME solutions.
- You MUST provide a final answer as an integer.
- Write your final answer in the format: \\boxed{{answer}}

Solve this AIME problem now:"""

def extract_answer(response):
    """Extract numerical answer from model response."""
    # Try to find \boxed{number}
    boxed_pattern = r'\\boxed\{([^}]+)\}'
    matches = re.findall(boxed_pattern, response)
    if matches:
        # Get the last boxed answer (usually the final answer)
        answer = matches[-1].strip()
        # Extract just the number if there's extra text
        num_match = re.search(r'(-?\d+)', answer)
        if num_match:
            return num_match.group(1)
        return answer

    # Fallback: try to find "answer is X" or "= X" at the end
    answer_patterns = [
        r'answer\s*(?:is|=|:)\s*(-?\d+)',
        r'final\s*answer\s*(?:is|=|:)\s*(-?\d+)',
        r'=\s*(-?\d+)\s*$',
        r'(-?\d+)\s*$'
    ]

    for pattern in answer_patterns:
        match = re.search(pattern, response, re.IGNORECASE | re.MULTILINE)
        if match:
            return match.group(1)

    return None

def check_answer(predicted, correct):
    """Check if predicted answer matches correct answer."""
    if predicted is None:
        return False
    try:
        return int(predicted) == int(correct)
    except (ValueError, TypeError):
        return str(predicted).strip() == str(correct).strip()

## 5. Parallel Batch Evaluation Function

In [ ]:
def evaluate_model_parallel_batch(model_name, model, problems, variation_type, num_rollouts=10, batch_size=30):
    """Evaluate a model with parallel batch processing.
    
    Args:
        model_name: Model name for display
        model: vLLM model instance
        problems: List of problem dictionaries
        variation_type: 'node_deletion_problem' or 'edge_deletion_problem'
        num_rollouts: Number of rollouts per problem (default: 10)
        batch_size: Number of questions to process in parallel (default: 30)
    """
    all_results = []
    
    print(f"\n{'='*80}")
    print(f"Evaluating {model_name} - {variation_type}")
    print(f"Batch size: {batch_size} questions processed in parallel")
    print(f"Rollouts per question: {num_rollouts}")
    print(f"Total problems: {len(problems)}")
    print(f"{'='*80}")
    
    # Process in batches
    for batch_start in range(0, len(problems), batch_size):
        batch_end = min(batch_start + batch_size, len(problems))
        batch_problems = problems[batch_start:batch_end]
        
        print(f"\nProcessing batch {batch_start//batch_size + 1}: Problems {batch_start+1} to {batch_end}")
        
        # Prepare all prompts for this batch
        prompts = []
        for prob in batch_problems:
            problem_text = prob[variation_type]
            
            # Create AIME prompt WITHOUT year/problem number
            prompt = f"""{AIME_SYSTEM_PROMPT}

{create_aime_prompt(problem_text)}"""
            prompts.append(prompt)
        
        # vLLM sampling parameters for multiple rollouts
        # Input: ~991 tokens, Output: 2048 tokens, Total: ~3039 tokens (fits in 4096!)
        sampling_params = SamplingParams(
            temperature=0.6,
            top_p=0.9,
            max_tokens=2048,  # Increased back to 2048 - safe with 4096 context
            n=num_rollouts,  # Generate num_rollouts responses per prompt
            stop=["\n\n\n"]  # Stop at triple newline to prevent excessive generation
        )
        
        # Generate all responses for all problems in this batch
        print(f"  Generating {len(prompts)} × {num_rollouts} = {len(prompts) * num_rollouts} responses in parallel...")
        try:
            outputs = model.generate(prompts, sampling_params)
        except Exception as e:
            print(f"  ✗ Batch generation error: {str(e)[:200]}")
            print(f"  Full error:")
            import traceback
            traceback.print_exc()
            # Create dummy outputs for error handling
            outputs = []
            for _ in range(len(prompts)):
                class DummyOutput:
                    def __init__(self):
                        self.outputs = [type('obj', (object,), {'text': f"Error: {str(e)}"})() for _ in range(num_rollouts)]
                outputs.append(DummyOutput())
        
        # Process results for each problem
        for prob_idx, (prob, output) in enumerate(zip(batch_problems, outputs)):
            correct_answer = prob['original_answer']
            responses = [o.text for o in output.outputs]
            
            # Process all rollouts
            all_predictions = []
            first_correct_attempt = None
            
            for attempt, response in enumerate(responses, 1):
                try:
                    predicted = extract_answer(response)
                    is_correct = check_answer(predicted, correct_answer)
                    all_predictions.append(predicted)
                    
                    # Track first correct attempt
                    if is_correct and first_correct_attempt is None:
                        first_correct_attempt = attempt
                except Exception as e:
                    all_predictions.append(None)
            
            # Determine final result
            if first_correct_attempt is not None:
                is_correct = True
                attempt_number = first_correct_attempt
                predicted_answer = all_predictions[first_correct_attempt - 1]
                status = '✓'
            else:
                is_correct = False
                attempt_number = num_rollouts
                predicted_answer = all_predictions[-1]
                status = '✗'
            
            print(f"  Problem {prob['problem_num']} ({prob['split']}): {status} Predicted={predicted_answer}, Correct={correct_answer}, First correct at rollout {first_correct_attempt if is_correct else 'N/A'}")
            
            # Store result
            result_dict = {
                'model': model_name,
                'split': prob['split'],
                'year': prob['year'],
                'problem_num': prob['problem_num'],
                'problem_idx': prob['problem_idx'],
                'variation': variation_type,
                'problem': prob[variation_type][:500],
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'is_correct': is_correct,
                'attempts': attempt_number,
                'full_response': responses[-1]
            }
            
            # Add individual rollout predictions AND full raw responses
            for i in range(num_rollouts):
                result_dict[f'rollout_{i+1}_prediction'] = all_predictions[i] if i < len(all_predictions) else None
                result_dict[f'rollout_{i+1}_response'] = responses[i] if i < len(responses) else None
            
            all_results.append(result_dict)
    
    return all_results

## 6. Run Evaluation (Models Loaded One at a Time)

This section:
1. Loads each model individually
2. Evaluates it on all variations
3. Unloads the model and clears GPU memory
4. Repeats for the next model

This approach keeps GPU memory usage under 44GB by only having one model loaded at a time.

In [ ]:
# Test configuration
test_model_configs = [
    ('DeepSeek-LLM-7B-Chat', 'deepseek-ai/deepseek-llm-7b-chat', 4096),
    ('DeepSeek-Math-7B-Instruct', 'deepseek-ai/deepseek-math-7b-instruct', 4096),
    ('InternLM2-Math-7B', 'internlm/internlm2-math-7b', 4096),
    ('InternLM2-Chat-7B', 'internlm/internlm2-chat-7b', 4096),
    ('Qwen2.5-7B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', 4096),
    ('Qwen2.5-Math-7B-Instruct', 'Qwen/Qwen2.5-Math-7B-Instruct', 4096)
]

# Use first problem as test
test_problem = all_problems[0]
test_variation = 'node_deletion_problem'

print("=" * 80)
print("TESTING ALL MODELS WITH SINGLE PROBLEM")
print("=" * 80)
print(f"\nTest Problem: {test_problem['split']} #{test_problem['problem_num']}")
print(f"Variation: {test_variation}")
print(f"Correct Answer: {test_problem['original_answer']}")
print(f"\nProblem Text: {test_problem[test_variation][:200]}...")

test_results = []

# Test each model
for model_idx, (model_name, model_path, max_len) in enumerate(test_model_configs, 1):
    print(f"\n{'='*80}")
    print(f"[{model_idx}/{len(test_model_configs)}] TESTING: {model_name}")
    print(f"{'='*80}")
    
    test_start_time = __import__('time').time()
    
    try:
        # Step 1: Load model
        print(f"\n1️⃣ Loading model...")
        print(f"   Path: {model_path}")
        print(f"   Context Length: {max_len}")
        
        model = LLM(
            model=model_path,
            trust_remote_code=True,
            dtype="bfloat16",
            max_model_len=max_len,
            tensor_parallel_size=1,
            gpu_memory_utilization=0.85,
            enforce_eager=False,
            download_dir=None,
        )
        
        print(f"   ✓ Model loaded successfully!")
        
        # Step 2: Generate test response
        print(f"\n2️⃣ Generating response (1 rollout)...")
        
        problem_text = test_problem[test_variation]
        
        # Create AIME prompt WITHOUT year/problem number
        test_prompt = f"""{AIME_SYSTEM_PROMPT}

{create_aime_prompt(problem_text)}"""
        
        sampling_params = SamplingParams(
            temperature=0.6,
            top_p=0.9,
            max_tokens=2048,  # Increased to match evaluation
            n=1,
            stop=["\n\n\n"]
        )
        
        outputs = model.generate([test_prompt], sampling_params)
        response = outputs[0].outputs[0].text
        
        print(f"   ✓ Generation successful!")
        print(f"   Response length: {len(response)} characters")
        
        # Step 3: Extract and check answer
        print(f"\n3️⃣ Extracting answer...")
        
        predicted = extract_answer(response)
        is_correct = check_answer(predicted, test_problem['original_answer'])
        
        print(f"   Predicted: {predicted}")
        print(f"   Correct: {test_problem['original_answer']}")
        print(f"   Result: {'✓ CORRECT' if is_correct else '✗ INCORRECT'}")
        
        # Step 4: Unload model
        print(f"\n4️⃣ Unloading model and clearing GPU memory...")
        
        del model
        gc.collect()
        torch.cuda.empty_cache()
        
        print(f"   ✓ Model unloaded successfully!")
        
        # Calculate test time
        test_time = __import__('time').time() - test_start_time
        
        # Store result
        test_results.append({
            'model': model_name,
            'status': '✓ SUCCESS',
            'predicted': predicted,
            'correct': is_correct,
            'time': f"{test_time:.1f}s"
        })
        
        print(f"\n{'='*80}")
        print(f"✓ {model_name} TEST PASSED ({test_time:.1f}s)")
        print(f"{'='*80}")
        
    except Exception as e:
        print(f"\n{'='*80}")
        print(f"✗ {model_name} TEST FAILED")
        print(f"{'='*80}")
        print(f"\nError: {str(e)}")
        print(f"\nFull traceback:")
        import traceback
        traceback.print_exc()
        
        # Store failure
        test_results.append({
            'model': model_name,
            'status': f'✗ FAILED: {str(e)[:50]}...',
            'predicted': None,
            'correct': False,
            'time': 'N/A'
        })
        
        # Try to clean up
        try:
            del model
        except:
            pass
        gc.collect()
        torch.cuda.empty_cache()
        
        print(f"\nContinuing to next model...")

# Print summary
print(f"\n\n{'='*80}")
print(f"TEST SUMMARY")
print(f"{'='*80}")
print(f"\nTested {len(test_results)} models:\n")

for result in test_results:
    status_icon = '✓' if 'SUCCESS' in result['status'] else '✗'
    print(f"{status_icon} {result['model']:<35} {result['status']:<20} Time: {result['time']}")

passed = sum(1 for r in test_results if 'SUCCESS' in r['status'])
failed = len(test_results) - passed

print(f"\n{'='*80}")
if failed == 0:
    print(f"🎉 ALL {len(test_results)} MODELS PASSED!")
    print(f"You can now run the full evaluation safely.")
else:
    print(f"⚠️  {passed}/{len(test_results)} models passed, {failed} failed")
    print(f"Fix the failed models before running full evaluation.")
print(f"{'='*80}")

## 6.1 Test Run - Verify All Models Load Correctly

This cell tests each model with a single problem to verify:
- Model loads successfully
- vLLM inference works
- Context length is appropriate
- GPU memory is sufficient
- Model unloads properly

**Run this first before the full evaluation!**

In [ ]:
# Store all results
all_results = []

# Define models to evaluate (load one at a time)
# Model configs with appropriate context lengths
model_configs = [
    ('DeepSeek-LLM-7B-Chat', 'deepseek-ai/deepseek-llm-7b-chat', 4096),
    ('DeepSeek-Math-7B-Instruct', 'deepseek-ai/deepseek-math-7b-instruct', 4096),
    ('InternLM2-Math-7B', 'internlm/internlm2-math-7b', 4096),
    ('InternLM2-Chat-7B', 'internlm/internlm2-chat-7b', 4096),
    ('Qwen2.5-7B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', 4096),
    ('Qwen2.5-Math-7B-Instruct', 'Qwen/Qwen2.5-Math-7B-Instruct', 4096)
]

# Define variations
variations = ['node_deletion_problem', 'edge_deletion_problem']

# Evaluate each model one at a time
for model_name, model_path, max_len in model_configs:
    print(f"\n{'='*80}")
    print(f"LOADING MODEL: {model_name}")
    print(f"Model Path: {model_path}")
    print(f"Max Context Length: {max_len}")
    print(f"{'='*80}")
    
    # Load the model
    print(f"Loading {model_path} with vLLM...")
    try:
        model = LLM(
            model=model_path,
            trust_remote_code=True,
            dtype="bfloat16",
            max_model_len=max_len,  # Use model-specific context length
            tensor_parallel_size=1,
            gpu_memory_utilization=0.85,  # Reduced from 0.9 for stability
            enforce_eager=False,  # Use CUDA graphs for better performance
            download_dir=None,  # Use default cache
        )
        print(f"✓ {model_name} loaded successfully!")
        
        # Run evaluation for all variations
        for variation in variations:
            print(f"\nEvaluating {variation}...")
            try:
                results = evaluate_model_parallel_batch(
                    model_name=model_name,
                    model=model,
                    problems=all_problems,
                    variation_type=variation,
                    num_rollouts=10,
                    batch_size=30
                )
                all_results.extend(results)
                print(f"✓ {variation} evaluation complete: {len(results)} results")
            except Exception as eval_error:
                print(f"✗ Error evaluating {variation}: {str(eval_error)}")
                print("Continuing to next variation...")
                continue
        
        # Free GPU memory
        print(f"\n{'='*80}")
        print(f"UNLOADING MODEL: {model_name}")
        print(f"{'='*80}")
        del model
        gc.collect()
        torch.cuda.empty_cache()
        print(f"✓ {model_name} unloaded and GPU memory cleared")
        
    except Exception as e:
        print(f"✗ Error loading or evaluating {model_name}: {str(e)}")
        print(f"Full error details:")
        import traceback
        traceback.print_exc()
        print(f"\nSkipping {model_name} and moving to next model...")
        # Try to free memory even if there was an error
        try:
            del model
        except:
            pass
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n{'='*80}")
print(f"ALL EVALUATIONS COMPLETE!")
print(f"Total results: {len(all_results)}")
print(f"{'='*80}")
print(f"Models evaluated: {len(set([r['model'] for r in all_results]))}")
print(f"Results per model: {len(all_results) // len(set([r['model'] for r in all_results])) if all_results else 0}")

## 7. Results Summary and Analysis

In [ ]:
def print_summary(results):
    """Print evaluation summary with clean table format."""
    df = pd.DataFrame(results)

    # Create summary table - Questions solved (counting unique questions, not variations)
    print("\n" + "="*80)
    print("--- SUMMARY TABLE (Pass@10 - Final Results After 10 Rollouts) ---")
    print("="*80)
    
    summary_data = []
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        
        # Overall: count unique questions where at least one variation was solved
        # Group by both split and problem_num to handle duplicate problem numbers across splits
        questions_solved = model_df[model_df['is_correct'] == True].groupby(['split', 'problem_num']).size()
        total_questions = model_df.groupby(['split', 'problem_num']).size().count()
        overall_solved = len(questions_solved)
        overall_acc = (overall_solved / total_questions * 100) if total_questions > 0 else 0
        
        # AIME24: count unique questions solved
        aime24_df = model_df[model_df['split'] == 'AIME24']
        aime24_questions_solved = aime24_df[aime24_df['is_correct'] == True].groupby('problem_num').size()
        aime24_total = aime24_df.groupby('problem_num').size().count() if len(aime24_df) > 0 else 0
        aime24_solved = len(aime24_questions_solved) if len(aime24_df) > 0 else 0
        aime24_acc = (aime24_solved / aime24_total * 100) if aime24_total > 0 else 0
        
        # AIME25: count unique questions solved
        aime25_df = model_df[model_df['split'] == 'AIME25']
        aime25_questions_solved = aime25_df[aime25_df['is_correct'] == True].groupby('problem_num').size()
        aime25_total = aime25_df.groupby('problem_num').size().count() if len(aime25_df) > 0 else 0
        aime25_solved = len(aime25_questions_solved) if len(aime25_df) > 0 else 0
        aime25_acc = (aime25_solved / aime25_total * 100) if aime25_total > 0 else 0
        
        summary_data.append({
            'Model': model,
            'Accuracy (%)': f"{overall_solved}/{total_questions} ({overall_acc:.1f}%)",
            'AIME24 (%)': f"{aime24_solved}/{aime24_total} ({aime24_acc:.1f}%)",
            'AIME25 (%)': f"{aime25_solved}/{aime25_total} ({aime25_acc:.1f}%)"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=True))
    print("\nNote: A question is counted as solved if ANY variation (node or edge deletion)")
    print("      was solved in ANY of the 10 rollouts.")

    print("\n" + "="*80)
    print("DETAILED EVALUATION SUMMARY")
    print("="*80)

    # Overall results by model
    print("\n--- Results by Model ---\n")
    
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        
        print(f"\n{model}:")
        
        # Node deletion
        node_df = model_df[model_df['variation'] == 'node_deletion_problem']
        node_acc = node_df['is_correct'].mean() * 100 if len(node_df) > 0 else 0
        node_correct = node_df['is_correct'].sum() if len(node_df) > 0 else 0
        node_total = len(node_df) if len(node_df) > 0 else 0
        node_avg_attempts = node_df['attempts'].mean() if len(node_df) > 0 else 0
        
        # Edge deletion
        edge_df = model_df[model_df['variation'] == 'edge_deletion_problem']
        edge_acc = edge_df['is_correct'].mean() * 100 if len(edge_df) > 0 else 0
        edge_correct = edge_df['is_correct'].sum() if len(edge_df) > 0 else 0
        edge_total = len(edge_df) if len(edge_df) > 0 else 0
        edge_avg_attempts = edge_df['attempts'].mean() if len(edge_df) > 0 else 0
        
        # Total
        total_acc = model_df['is_correct'].mean() * 100
        total_correct = model_df['is_correct'].sum()
        total_total = len(model_df)
        total_avg_attempts = model_df['attempts'].mean()
        
        print(f"  Node Deletion: {node_correct}/{node_total} ({node_acc:.1f}%) - Avg attempts: {node_avg_attempts:.1f}")
        print(f"  Edge Deletion: {edge_correct}/{edge_total} ({edge_acc:.1f}%) - Avg attempts: {edge_avg_attempts:.1f}")
        print(f"  Total: {total_correct}/{total_total} ({total_acc:.1f}%) - Avg attempts: {total_avg_attempts:.1f}")

    # Overall comparison
    print("\n" + "="*80)
    print("--- Overall Model Comparison ---")
    print("="*80)
    model_acc = df.groupby('model')['is_correct'].mean() * 100
    for model, acc in sorted(model_acc.items(), key=lambda x: x[1], reverse=True):
        total = len(df[df['model'] == model])
        correct = df[df['model'] == model]['is_correct'].sum()
        avg_attempts = df[df['model'] == model]['attempts'].mean()
        print(f"{model}: {correct}/{total} ({acc:.1f}%) - Avg attempts: {avg_attempts:.1f}")

    # Attempt distribution for correct answers
    print("\n" + "="*80)
    print("--- Attempt Distribution (for correct answers) ---")
    print("="*80)
    for model in df['model'].unique():
        model_correct = df[(df['model'] == model) & (df['is_correct'] == True)]
        if len(model_correct) > 0:
            print(f"\n{model}:")
            attempt_counts = model_correct['attempts'].value_counts().sort_index()
            for attempt, count in attempt_counts.items():
                print(f"  Rollout {attempt}: {count} problems")
        else:
            print(f"\n{model}: No correct answers")

    # Pass@k statistics
    print("\n" + "="*80)
    print("--- Pass@k Statistics ---")
    print("="*80)
    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        print(f"\n{model}:")
        for k in [1, 3, 5, 10]:
            # Count problems where correct answer was found in first k attempts
            pass_at_k = 0
            for _, row in model_df.iterrows():
                if row['is_correct'] and row['attempts'] <= k:
                    pass_at_k += 1
            pass_at_k_rate = (pass_at_k / len(model_df)) * 100
            print(f"  Pass@{k}: {pass_at_k}/{len(model_df)} ({pass_at_k_rate:.1f}%)")

    return df

# Print summary
results_df = print_summary(all_results)

## 8. Save Results

In [ ]:
# Create output folder
import os
import json
import numpy as np

output_folder = 'varevalpass10_results'
os.makedirs(output_folder, exist_ok=True)

# Helper function to convert numpy types to Python native types
def convert_to_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    return obj

# Save complete CSV file with all results
csv_path = os.path.join(output_folder, 'all_results.csv')
results_df.to_csv(csv_path, index=False)
print(f"✓ Saved complete CSV: {csv_path}")

# Save complete JSON file with all results
json_path = os.path.join(output_folder, 'all_results.json')
all_results_json = convert_to_serializable(results_df.to_dict('records'))
with open(json_path, 'w') as f:
    json.dump(all_results_json, f, indent=2)
print(f"✓ Saved complete JSON: {json_path}")

# Save individual JSON files per model
for model in results_df['model'].unique():
    model_df = results_df[results_df['model'] == model]
    
    # Convert to list of dictionaries for JSON
    model_results = model_df.to_dict('records')
    
    # Convert numpy types to Python types
    model_results = convert_to_serializable(model_results)
    
    # Clean model name for filename
    model_filename = model.replace('/', '_').replace('.', '_').replace('-', '_')
    model_json_path = os.path.join(output_folder, f'{model_filename}.json')
    
    with open(model_json_path, 'w') as f:
        json.dump(model_results, f, indent=2)
    
    print(f"  - Saved {len(model_results)} results for {model}")

# Save the summary table
summary_data = []
for model in results_df['model'].unique():
    model_df = results_df[results_df['model'] == model]
    
    # Overall: count unique questions where at least one variation was solved
    questions_solved = model_df[model_df['is_correct'] == True].groupby(['split', 'problem_num']).size()
    total_questions = model_df.groupby(['split', 'problem_num']).size().count()
    overall_solved = len(questions_solved)
    overall_acc = (overall_solved / total_questions * 100) if total_questions > 0 else 0
    
    # AIME24
    aime24_df = model_df[model_df['split'] == 'AIME24']
    aime24_questions_solved = aime24_df[aime24_df['is_correct'] == True].groupby('problem_num').size()
    aime24_total = aime24_df.groupby('problem_num').size().count() if len(aime24_df) > 0 else 0
    aime24_solved = len(aime24_questions_solved) if len(aime24_df) > 0 else 0
    aime24_acc = (aime24_solved / aime24_total * 100) if aime24_total > 0 else 0
    
    # AIME25
    aime25_df = model_df[model_df['split'] == 'AIME25']
    aime25_questions_solved = aime25_df[aime25_df['is_correct'] == True].groupby('problem_num').size()
    aime25_total = aime25_df.groupby('problem_num').size().count() if len(aime25_df) > 0 else 0
    aime25_solved = len(aime25_questions_solved) if len(aime25_df) > 0 else 0
    aime25_acc = (aime25_solved / aime25_total * 100) if aime25_total > 0 else 0
    
    summary_data.append({
        'model': model,
        'overall': {'solved': int(overall_solved), 'total': int(total_questions), 'accuracy': round(overall_acc, 1)},
        'aime24': {'solved': int(aime24_solved), 'total': int(aime24_total), 'accuracy': round(aime24_acc, 1)},
        'aime25': {'solved': int(aime25_solved), 'total': int(aime25_total), 'accuracy': round(aime25_acc, 1)}
    })

summary_json_path = os.path.join(output_folder, 'summary.json')
with open(summary_json_path, 'w') as f:
    json.dump(summary_data, f, indent=2)
print(f"✓ Saved summary JSON: {summary_json_path}")

# Save summary as CSV too
summary_csv_data = []
for item in summary_data:
    summary_csv_data.append({
        'Model': item['model'],
        'Overall_Solved': item['overall']['solved'],
        'Overall_Total': item['overall']['total'],
        'Overall_Accuracy': item['overall']['accuracy'],
        'AIME24_Solved': item['aime24']['solved'],
        'AIME24_Total': item['aime24']['total'],
        'AIME24_Accuracy': item['aime24']['accuracy'],
        'AIME25_Solved': item['aime25']['solved'],
        'AIME25_Total': item['aime25']['total'],
        'AIME25_Accuracy': item['aime25']['accuracy']
    })

summary_csv_path = os.path.join(output_folder, 'summary.csv')
pd.DataFrame(summary_csv_data).to_csv(summary_csv_path, index=False)
print(f"✓ Saved summary CSV: {summary_csv_path}")

print(f"\n{'='*80}")
print(f"ALL FILES SAVED TO: {os.path.abspath(output_folder)}")
print(f"{'='*80}")
print(f"  📊 all_results.csv - Complete results in CSV format")
print(f"  📄 all_results.json - Complete results in JSON format")
print(f"  📋 summary.csv - Summary accuracies in CSV format")
print(f"  📋 summary.json - Summary accuracies in JSON format")
print(f"  📁 {len(results_df['model'].unique())} individual model JSON files")
print(f"{'='*80}")

## 9. View Detailed Results

In [ ]:
# View only CORRECT predictions
correct = results_df[results_df['is_correct'] == True]
print(f"Total correct predictions: {len(correct)}")

if len(correct) > 0:
    print("\n" + "="*80)
    print("CORRECT PREDICTIONS")
    print("="*80)
    print(correct[['model', 'split', 'variation', 'problem_num', 'predicted_answer', 'correct_answer', 'attempts']].to_string(index=False))
else:
    print("\nNo correct predictions found.")

In [ ]:
# Breakdown by model - only show correct predictions
print("\n" + "="*80)
print("CORRECT PREDICTIONS BY MODEL")
print("="*80)

for model in results_df['model'].unique():
    model_correct = results_df[(results_df['model'] == model) & (results_df['is_correct'] == True)]
    print(f"\n{model}: {len(model_correct)} correct")
    
    if len(model_correct) > 0:
        for _, row in model_correct.iterrows():
            print(f"  ✓ {row['split']} Problem {row['problem_num']} ({row['variation'].replace('_problem', '')})")
            print(f"    Answer: {row['predicted_answer']} | Attempt: {row['attempts']}")
    else:
        print("  No correct predictions")